# AetherLab Python SDK 0.5.2 — End-to-End Guardrails

This notebook teaches the **published `aetherlab==0.5.2` API** against the production base URL `https://api.aetherlab.co`. It covers scalar PromptGuard and MediaGuard checks, two server-side batch approaches for each guardrail, result correlation, resource inspection, and opt-in cleanup.

> **Production and cost warning:** Setting `RUN_LIVE_TESTS = True` sends content to AetherLab's production API, can incur usage charges, and creates batch/file resources that remain until retention expiry or explicit cleanup. The notebook defaults to safe, no-call mode and uses only one benign item per batch.

## Notebook sections

1. Overview, requirements, and safe credentials
2. Install/import SDK 0.5.2 and create the client
3. PromptGuard scalar inference
4. MediaGuard scalar inference
5. PromptGuard batch — inline helper
6. PromptGuard batch — uploaded UTF-8 JSONL
7. MediaGuard batch — inline HTTPS URL
8. MediaGuard batch — uploaded UTF-8 JSONL
9. Resource inspection and optional cleanup
10. Troubleshooting and semantics

## 1. Overview, requirements, and safe credential setup

### Requirements

- Python 3.9 or newer and a Jupyter kernel that can install packages.
- An AetherLab API key from [app.aetherlab.co](https://app.aetherlab.co) for live execution.
- Network access to `https://api.aetherlab.co` and the public HTTPS image used below.

### How `export` works

Run this in a **terminal before launching Jupyter** so the Jupyter server and its kernels inherit the variable:

```bash
export AETHERLAB_API_KEY="<your-key>"
jupyter lab
```

`export AETHERLAB_API_KEY=...` changes the environment of that terminal process and future child processes. It does not edit Python or the notebook. By contrast, `!export AETHERLAB_API_KEY=...` does **not** persist between notebook cells: each `!` command runs in a separate shell subprocess, and a child process cannot change the parent Python kernel's environment.

Recommended notebook-safe choices:

1. **Inherited environment variable:** export in the terminal, then launch Jupyter from that same terminal.
2. **Notebook-native prompt:** let `getpass.getpass()` request the key only in kernel memory. The input is hidden and is not written into cell source or output.

Never paste a real key into a code cell, command, output, notebook metadata, or saved file. Clear all outputs before sharing an executed notebook. This notebook never prints the key or the full environment.

In [ ]:
# Safety gate: leave False for a no-cost, no-network walkthrough.
# To run every production inference and batch path, change this one value to True,
# then restart the kernel and run all cells from the top.
RUN_LIVE_TESTS = False

print(
    "Live production calls are ENABLED." if RUN_LIVE_TESTS
    else "Safe mode: production calls are disabled. Set RUN_LIVE_TESTS = True to opt in."
)

## 2. Install/import `aetherlab==0.5.2` and create the client

`%pip` installs into the active Jupyter kernel's Python environment. Pinning `0.5.2` keeps every signature and result model in this tutorial reproducible, including MediaGuard `industry` forwarding and mapped prompt items that use the documented `input` alias.

The client is created only when live mode is enabled. It receives the key from the inherited `AETHERLAB_API_KEY` variable, or from a hidden `getpass.getpass()` prompt if the variable is absent. Neither path displays the key.

In [ ]:
%pip install --quiet "aetherlab==0.5.2"

In [ ]:
import getpass
import json
import os
import tempfile
import uuid
from pathlib import Path

from IPython.display import display

import aetherlab
from aetherlab import AetherLabClient

EXPECTED_SDK_VERSION = "0.5.2"
API_BASE_URL = "https://api.aetherlab.co"
MEDIA_URL = "https://raw.githubusercontent.com/github/explore/main/topics/python/python.png"

if aetherlab.__version__ != EXPECTED_SDK_VERSION:
    raise RuntimeError(
        f"Expected aetherlab {EXPECTED_SDK_VERSION}, found {aetherlab.__version__}. "
        "Restart the kernel after the pinned install cell."
    )

# Keep the key only in kernel memory. Never print this variable.
_api_key = None
if RUN_LIVE_TESTS:
    _api_key = os.environ.get("AETHERLAB_API_KEY")
    if not _api_key:
        _api_key = getpass.getpass("AetherLab API key (hidden; kept only in kernel memory): ")

client = (
    AetherLabClient(
        api_key=_api_key,
        base_url=API_BASE_URL,
        timeout=120.0,
        max_retries=3,
    )
    if RUN_LIVE_TESTS
    else None
)

# Stable for this kernel run; restart the kernel to intentionally create a new run.
RUN_TOKEN = globals().get("RUN_TOKEN") or uuid.uuid4().hex[:12]
RUN_IDS = {
    "prompt_inline_custom": f"prompt-inline-{RUN_TOKEN}-01",
    "prompt_inline_key": f"notebook-{RUN_TOKEN}-prompt-inline",
    "prompt_jsonl_custom": f"prompt-jsonl-{RUN_TOKEN}-01",
    "prompt_jsonl_key": f"notebook-{RUN_TOKEN}-prompt-jsonl",
    "media_inline_custom": f"media-inline-{RUN_TOKEN}-01",
    "media_inline_key": f"notebook-{RUN_TOKEN}-media-inline",
    "media_jsonl_custom": f"media-jsonl-{RUN_TOKEN}-01",
    "media_jsonl_key": f"notebook-{RUN_TOKEN}-media-jsonl",
}

CREATED_BATCHES = {}
UPLOADED_FILES = {}
LOCAL_JSONL_PATHS = []
VALIDATION_ISSUES = {}


def format_compliance(result):
    """Select the public ComplianceResult fields without dumping raw payloads."""
    return {
        "compliance_status": result.compliance_status,
        "is_compliant": result.is_compliant,
        "avg_threat_level": result.avg_threat_level,
        "confidence": result.confidence,
        "rationale": result.rationale,
    }


def format_batch_job(job):
    """Format BatchJob metadata. Creation metadata is not an item verdict."""
    counts = None
    if job.request_counts is not None:
        counts = {
            "total": job.request_counts.total,
            "pending": job.request_counts.pending,
            "in_progress": job.request_counts.in_progress,
            "succeeded": job.request_counts.succeeded,
            "failed": job.request_counts.failed,
            "cancelled": job.request_counts.cancelled,
            "expired": job.request_counts.expired,
        }
    return {
        "batch_id": job.id,
        "endpoint": job.endpoint,
        "status": job.status,
        "completion_window": job.completion_window,
        "input_file_id": job.input_file_id,
        "request_counts": counts,
    }


def format_batch_item(item):
    """Format one unordered BatchResultItem and preserve its custom_id."""
    formatted = {
        "custom_id": item.custom_id,
        "item_status": item.status,
    }
    if item.result is not None:
        formatted["result"] = format_compliance(item.result)
    if item.error is not None:
        formatted["error"] = {
            "code": item.error.code,
            "message": item.error.message,
        }
    return formatted


def fetch_all_batch_results(batch):
    """Retrieve every page and correlate results by custom_id, never by order."""
    by_custom_id = {}
    after = None
    while True:
        page = client.get_batch_results(batch.id, limit=100, after=after)
        for item in page.items:
            if item.custom_id in by_custom_id:
                raise RuntimeError(f"Duplicate result custom_id: {item.custom_id}")
            by_custom_id[item.custom_id] = format_batch_item(item)
        if not page.has_more:
            break
        if not page.next_cursor:
            raise RuntimeError("The API reported more results without a next cursor.")
        after = page.next_cursor
    return dict(sorted(by_custom_id.items()))


def record_issue(label, exc):
    """Record an issue while defensively redacting the in-memory key."""
    message = str(exc)
    if _api_key:
        message = message.replace(_api_key, "[REDACTED]")
    VALIDATION_ISSUES[label] = {
        "exception_type": type(exc).__name__,
        "message": message,
    }
    print(f"{label} issue: {type(exc).__name__}: {message}")


print(f"aetherlab version: {aetherlab.__version__}")
print(f"API base URL: {API_BASE_URL}")
print(f"Public media URL: {MEDIA_URL}")
print(f"Per-run ID token: {RUN_TOKEN}")
print("Client state:", "ready for production calls" if client else "not created (safe mode)")

## 3. Single PromptGuard inference

This benign prompt is checked against an explicit policy (`blacklisted_keywords=["malware instructions"]`). Passing a policy makes the example independent of dashboard policy configuration and avoids `MissingPolicyError`.

A request can execute successfully and still return either **Compliant** or **Non-Compliant**. Those are both valid guardrail verdicts, not transport statuses.

In [ ]:
prompt_result = None

if RUN_LIVE_TESTS:
    try:
        prompt_result = client.check_prompt(
            "nude person",
            blacklisted_keywords=[""],
            whitelisted_keywords=["nsfw"],
            reasoning_mode="low",
            environment="production",
            timeout=120.0,
        )
        display(format_compliance(prompt_result))
    except Exception as exc:
        record_issue("PromptGuard scalar", exc)
else:
    print("Skipped PromptGuard scalar production call.")

## 4. Single MediaGuard inference

The SDK accepts an HTTPS image URL with `input_type="url"`. The next cell keeps its editable URL beside the scalar call so it is clear which value is sent.

> To test a different image, replace only `MEDIA_TEST_URL` in the next cell. Do not put a URL in `media_result`; that variable is reserved for the returned verdict.

The saved default is a benign public PNG and uses `blacklisted_keywords=["graphic violence"]`. For the two alternative policy cases, choose one policy style:

- Explicit keyword: use `blacklisted_keywords=["female"]`; any image matching that description is expected to be Non-Compliant.
- Industry preset: remove whitelist/blacklist arguments and use `industry="nsfw"`. This argument is available in `aetherlab==0.5.2`; published 0.5.1 omitted the production API field.

In [ ]:
# Replace only this value when testing another public HTTPS image.
MEDIA_TEST_URL = "https://raw.githubusercontent.com/github/explore/main/topics/python/python.png"
media_result = None

if RUN_LIVE_TESTS:
    try:
        media_result = client.check_media(
            MEDIA_TEST_URL,
            input_type="url",
            blacklisted_keywords=["graphic violence"],
            reasoning_mode="low",
            environment="production",
            timeout=180.0,
        )
        display(format_compliance(media_result))
    except Exception as exc:
        record_issue("MediaGuard scalar", exc)
else:
    print("Skipped MediaGuard scalar production call.")

## 5. PromptGuard batch — Approach 1: inline with `check_prompt_batch`

The convenience helper submits one server-side job. Its immediate return value is **job metadata**, not a compliance verdict. The required sequence is:

1. `check_prompt_batch(...)` to create the job.
2. `wait_for_batch(...)` to poll until a terminal job status.
3. `get_batch_results(...)` to retrieve item results.

Batch item order is not guaranteed. This notebook supplies a unique `custom_id` and correlates the result by that ID. The idempotency key is generated once per kernel run and reused if this cell is retried.

**SDK 0.5.2:** mapped prompt items accept `input`, `prompt`, or `user_prompt`. This example uses the recommended `input` form.

In [ ]:
prompt_inline_job = None
prompt_inline_results = {}

if RUN_LIVE_TESTS:
    try:
        prompt_inline_job = client.check_prompt_batch(
            [
                {
                    "custom_id": RUN_IDS["prompt_inline_custom"],
                    "input": "Please explain why regular backups are useful.",
                }
            ],
            idempotency_key=RUN_IDS["prompt_inline_key"],
            blacklisted_keywords=["malware instructions"],
            reasoning_mode="low",
            metadata={"tutorial": "sdk-0.5.2", "approach": "prompt-inline"},
        )
        CREATED_BATCHES["prompt_inline"] = prompt_inline_job
        print("Creation response (job metadata, not verdicts):")
        display(format_batch_job(prompt_inline_job))

        prompt_inline_job = client.wait_for_batch(
            prompt_inline_job,
            timeout=900.0,
            poll_interval=2.0,
            max_poll_interval=15.0,
        )
        CREATED_BATCHES["prompt_inline"] = prompt_inline_job
        print("Terminal job metadata:")
        display(format_batch_job(prompt_inline_job))

        prompt_inline_results = fetch_all_batch_results(prompt_inline_job)
        print("Item results keyed by custom_id (order-independent):")
        display(prompt_inline_results)
    except Exception as exc:
        record_issue("PromptGuard inline batch", exc)
else:
    print("Skipped PromptGuard inline batch production calls.")

## 6. PromptGuard batch — Approach 2: uploaded UTF-8 JSONL

JSONL is **newline-delimited JSON**: every non-empty line is one complete JSON request object. It is not a JSON array and there are no commas between lines.

This cell writes one benign request to a temporary UTF-8 `.jsonl` file, uploads it with `purpose="batch"`, then calls the generic `create_batch` method with the exact endpoint `/v1/guardrails/prompt`, `completion_window="24h"`, and a stable per-run idempotency key. The created file and batch are tracked for optional cleanup.

In [ ]:
prompt_jsonl_path = None
prompt_jsonl_file = None
prompt_jsonl_job = None
prompt_jsonl_results = {}

if RUN_LIVE_TESTS:
    try:
        prompt_request = {
            "custom_id": RUN_IDS["prompt_jsonl_custom"],
            "method": "POST",
            "url": "/v1/guardrails/prompt",
            "body": {
                "user_prompt": "Please describe a friendly way to welcome a new colleague.",
                "blacklisted_keyword": "malware instructions",
                "reasoning_mode": "low",
            },
        }
        with tempfile.NamedTemporaryFile(
            mode="w",
            encoding="utf-8",
            suffix=".jsonl",
            delete=False,
        ) as handle:
            handle.write(json.dumps(prompt_request, ensure_ascii=False) + "\n")
            prompt_jsonl_path = Path(handle.name)
        LOCAL_JSONL_PATHS.append(prompt_jsonl_path)
        print(f"Wrote one UTF-8 JSON object line to temporary file: {prompt_jsonl_path.name}")

        prompt_jsonl_file = client.upload_file(prompt_jsonl_path, purpose="batch")
        UPLOADED_FILES["prompt_jsonl"] = prompt_jsonl_file
        display({
            "file_id": prompt_jsonl_file.id,
            "filename": prompt_jsonl_file.filename,
            "purpose": prompt_jsonl_file.purpose,
            "bytes": prompt_jsonl_file.bytes,
        })

        prompt_jsonl_job = client.create_batch(
            "/v1/guardrails/prompt",
            input_file_id=prompt_jsonl_file.id,
            completion_window="24h",
            idempotency_key=RUN_IDS["prompt_jsonl_key"],
            metadata={"tutorial": "sdk-0.5.2", "approach": "prompt-jsonl"},
        )
        CREATED_BATCHES["prompt_jsonl"] = prompt_jsonl_job
        print("Creation response (job metadata, not verdicts):")
        display(format_batch_job(prompt_jsonl_job))

        prompt_jsonl_job = client.wait_for_batch(
            prompt_jsonl_job,
            timeout=900.0,
            poll_interval=2.0,
            max_poll_interval=15.0,
        )
        CREATED_BATCHES["prompt_jsonl"] = prompt_jsonl_job
        print("Terminal job metadata:")
        display(format_batch_job(prompt_jsonl_job))

        prompt_jsonl_results = fetch_all_batch_results(prompt_jsonl_job)
        print("Item results keyed by custom_id (order-independent):")
        display(prompt_jsonl_results)
    except Exception as exc:
        record_issue("PromptGuard JSONL batch", exc)
else:
    print("Skipped PromptGuard JSONL upload and batch production calls.")

## 7. MediaGuard batch — Approach 1: inline HTTPS URL items

`check_media_batch` accepts inline HTTPS URL items (or uploaded MediaGuard file IDs). Embedded base64 is deliberately unsupported for batch media.

As with PromptGuard, creation returns a `BatchJob`; wait for a terminal status, retrieve item results, and correlate them by `custom_id` rather than list position.

In [ ]:
media_inline_job = None
media_inline_results = {}

if RUN_LIVE_TESTS:
    try:
        media_inline_job = client.check_media_batch(
            [
                {
                    "custom_id": RUN_IDS["media_inline_custom"],
                    "url": MEDIA_URL,
                }
            ],
            idempotency_key=RUN_IDS["media_inline_key"],
            blacklisted_keywords=["graphic violence"],
            reasoning_mode="low",
            metadata={"tutorial": "sdk-0.5.2", "approach": "media-inline"},
        )
        CREATED_BATCHES["media_inline"] = media_inline_job
        print("Creation response (job metadata, not verdicts):")
        display(format_batch_job(media_inline_job))

        media_inline_job = client.wait_for_batch(
            media_inline_job,
            timeout=900.0,
            poll_interval=2.0,
            max_poll_interval=15.0,
        )
        CREATED_BATCHES["media_inline"] = media_inline_job
        print("Terminal job metadata:")
        display(format_batch_job(media_inline_job))

        media_inline_results = fetch_all_batch_results(media_inline_job)
        print("Item results keyed by custom_id (order-independent):")
        display(media_inline_results)
    except Exception as exc:
        record_issue("MediaGuard inline batch", exc)
else:
    print("Skipped MediaGuard inline batch production calls.")

## 8. MediaGuard batch — Approach 2: uploaded UTF-8 JSONL

The MediaGuard JSONL body must contain exactly one valid media reference. Here it uses `input_type="url"` plus the verified HTTPS image URL. Do not use base64, `data:` URLs, or plain HTTP in batch media.

The uploaded file again uses `purpose="batch"`; `purpose="guardrail_media"` is reserved for uploading an actual media file that will be referenced by file ID.

In [ ]:
media_jsonl_path = None
media_jsonl_file = None
media_jsonl_job = None
media_jsonl_results = {}

if RUN_LIVE_TESTS:
    try:
        media_request = {
            "custom_id": RUN_IDS["media_jsonl_custom"],
            "method": "POST",
            "url": "/v1/guardrails/media",
            "body": {
                "input_type": "url",
                "image": MEDIA_URL,
                "blacklisted_keyword": "graphic violence",
                "reasoning_mode": "low",
            },
        }
        with tempfile.NamedTemporaryFile(
            mode="w",
            encoding="utf-8",
            suffix=".jsonl",
            delete=False,
        ) as handle:
            handle.write(json.dumps(media_request, ensure_ascii=False) + "\n")
            media_jsonl_path = Path(handle.name)
        LOCAL_JSONL_PATHS.append(media_jsonl_path)
        print(f"Wrote one UTF-8 JSON object line to temporary file: {media_jsonl_path.name}")

        media_jsonl_file = client.upload_file(media_jsonl_path, purpose="batch")
        UPLOADED_FILES["media_jsonl"] = media_jsonl_file
        display({
            "file_id": media_jsonl_file.id,
            "filename": media_jsonl_file.filename,
            "purpose": media_jsonl_file.purpose,
            "bytes": media_jsonl_file.bytes,
        })

        media_jsonl_job = client.create_batch(
            "/v1/guardrails/media",
            input_file_id=media_jsonl_file.id,
            completion_window="24h",
            idempotency_key=RUN_IDS["media_jsonl_key"],
            metadata={"tutorial": "sdk-0.5.2", "approach": "media-jsonl"},
        )
        CREATED_BATCHES["media_jsonl"] = media_jsonl_job
        print("Creation response (job metadata, not verdicts):")
        display(format_batch_job(media_jsonl_job))

        media_jsonl_job = client.wait_for_batch(
            media_jsonl_job,
            timeout=900.0,
            poll_interval=2.0,
            max_poll_interval=15.0,
        )
        CREATED_BATCHES["media_jsonl"] = media_jsonl_job
        print("Terminal job metadata:")
        display(format_batch_job(media_jsonl_job))

        media_jsonl_results = fetch_all_batch_results(media_jsonl_job)
        print("Item results keyed by custom_id (order-independent):")
        display(media_jsonl_results)
    except Exception as exc:
        record_issue("MediaGuard JSONL batch", exc)
else:
    print("Skipped MediaGuard JSONL upload and batch production calls.")

## 9. Resource inspection and optional cleanup

The next cell retrieves current metadata for only the files and batches created by this kernel run. Remote resources are **not deleted by default**.

To remove created terminal batch resources, uploaded JSONL files, and local temporary JSONL files, set `CLEAN_UP_CREATED_RESOURCES = True` and rerun the cleanup cell. Batch resources must be terminal before deletion. Keep cleanup disabled if you need the IDs for later inspection.

In [ ]:
resource_snapshot = {"batches": {}, "files": {}}

if RUN_LIVE_TESTS:
    for label, job in CREATED_BATCHES.items():
        try:
            current = client.retrieve_batch(job.id)
            CREATED_BATCHES[label] = current
            resource_snapshot["batches"][label] = format_batch_job(current)
        except Exception as exc:
            record_issue(f"Inspect batch {label}", exc)

    for label, uploaded in UPLOADED_FILES.items():
        try:
            current = client.retrieve_file(uploaded.id)
            resource_snapshot["files"][label] = {
                "file_id": current.id,
                "filename": current.filename,
                "purpose": current.purpose,
                "bytes": current.bytes,
                "status": current.status,
                "expires_at": current.expires_at,
            }
        except Exception as exc:
            record_issue(f"Inspect file {label}", exc)

    print("Resources created by this notebook run:")
    display(resource_snapshot)
else:
    print("Skipped production resource inspection.")

In [ ]:
# Independent, explicit cleanup gate. Leave False to preserve remote resources.
CLEAN_UP_CREATED_RESOURCES = False

if RUN_LIVE_TESTS and CLEAN_UP_CREATED_RESOURCES:
    for label, job in list(CREATED_BATCHES.items()):
        try:
            current = client.retrieve_batch(job.id)
            if current.is_terminal:
                client.delete_batch(current)
                print(f"Deleted terminal batch {label}: {current.id}")
            else:
                print(f"Kept non-terminal batch {label}: {current.id} ({current.status})")
        except Exception as exc:
            record_issue(f"Clean up batch {label}", exc)

    for label, uploaded in list(UPLOADED_FILES.items()):
        try:
            client.delete_file(uploaded.id)
            print(f"Deleted uploaded file {label}: {uploaded.id}")
        except Exception as exc:
            record_issue(f"Clean up file {label}", exc)

    for path in LOCAL_JSONL_PATHS:
        try:
            path.unlink(missing_ok=True)
            print(f"Deleted local temporary JSONL: {path.name}")
        except Exception as exc:
            record_issue(f"Clean up local file {path.name}", exc)
elif RUN_LIVE_TESTS:
    print(
        "Cleanup disabled: remote batches/files and local JSONL files were retained. "
        "Set CLEAN_UP_CREATED_RESOURCES = True and rerun this cell to delete them."
    )
else:
    print("Safe mode: no production resources were created.")

if VALIDATION_ISSUES:
    print("Issues recorded during this run:")
    display(VALIDATION_ISSUES)
else:
    print("No SDK/API issues were recorded during this run.")

if client is not None:
    client.close()
    client = None
    _api_key = None
    print("AetherLab client closed; this notebook's local key variable was cleared.")

## 10. Troubleshooting and result semantics

### Environment variables and `export`

- Run `export AETHERLAB_API_KEY="<your-key>"` in a terminal **before** starting Jupyter from that terminal. The kernel inherits the variable at process creation.
- `!export ...` does not persist between cells because each `!` invocation uses a child shell subprocess. It cannot modify the parent kernel's environment.
- If the Jupyter server was already running when you exported the variable, restart it from the configured terminal, or use the hidden `getpass.getpass()` path in this notebook.
- Never diagnose credentials by printing the key or all of `os.environ`.

### Missing policy

PromptGuard and MediaGuard require at least one policy. Configure standing policies in Policy Controls or pass `whitelisted_keywords` / `blacklisted_keywords` with the request; MediaGuard can instead pass an enabled `industry` preset. Without a request or account policy, SDK 0.5.2 raises `MissingPolicyError` (`ERR_0202`). Every inference in this notebook passes an explicit benign test policy.

### Invalid or missing key

A missing key prevents client construction; an invalid key produces `AuthenticationError` / HTTP 401. Confirm the variable is inherited or re-enter it through the hidden prompt. Do not paste it into a cell.

### JSONL is not a JSON array

A batch input file must be UTF-8 JSONL: one object per non-empty line. Do not wrap objects in `[...]`, and do not place commas between lines. Every object needs a unique non-empty `custom_id` and an object-valued `body`.

### Media batch input

Media batch items must reference a public HTTPS URL or an uploaded `purpose="guardrail_media"` file ID. Embedded base64, `data:` URLs, and plain HTTP URLs are unsupported in batches. Scalar `check_media(..., input_type="base64")` is a separate supported path.

### Polling and timeouts

Batch creation only confirms queued job metadata. `wait_for_batch` polls with bounded backoff and can raise `TimeoutError` without proving the server job failed; retain the batch ID and inspect it later with `retrieve_batch`. Small jobs often take one or two minutes, but the processing window is not a latency SLA.

### Successful execution versus verdict

A batch item with `status == "succeeded"` completed technically. Its guardrail result may be either `Compliant` or `Non-Compliant`; both are successful evaluations. Item failures instead carry `status == "failed"` and an error. Always inspect every item and correlate by `custom_id`, because result order is unspecified.

### Completion window

SDK/API v0.5.2 currently accepts **only** `completion_window="24h"`. Other values are rejected before or during submission.